In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import norm


In [ ]:

import math
from pathlib import Path
from itertools import product, defaultdict

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from scipy.stats import norm

from src.on_off import (
    norm_survival,
    r_stat_onoff,
    u_stat_onoff,
    r_star_onoff,
    sample_null_toys,
    required_toys_for_Z_precision,
    pvals_onoff,
    asimov_Zs_onoff,
    median_expected_significance_onoff,
    expected_significance_onoff,
)


In [ ]:

# -------- P-value grids (writes plots/onoff_pval_plots.pdf) --------

# --- User inputs (edit here) ---
sigrel = 0.001
min_toys = 1_000
max_toys = 1_000_000

s_vec   = np.array([0.0, 2.0, 5.0])
b_vec   = np.array([0.5, 2.0, 5.0])
tau_vec = np.array([0.5, 1.0, 2.0])

out_dir  = Path("plots")
pdf_name = "onoff_pval_plots.pdf"

# -------------------------------------------------------------------

def nonneg_int_floor(x):
    return int(max(0, np.floor(x)))


def run_case(s0: float, b: float, tau: float, sigrel: float, pdf: PdfPages):
    mu_s = s0 + b
    mu_b = tau * b

    m0_raw = np.array([
        nonneg_int_floor(mu_b - np.sqrt(mu_b)),
        nonneg_int_floor(mu_b),
        nonneg_int_floor(mu_b + np.sqrt(mu_b)),
    ], dtype=int)

    m0_sorted = np.sort(m0_raw)
    m0_unique = np.unique(m0_sorted)

    if m0_unique.size == 1:
        k = m0_unique[0]
        m0_list = np.array([k, k + 1], dtype=int)
    else:
        m0_list = m0_unique

    nmax_raw = mu_s + 5.0 * np.sqrt(mu_s) + 2
    nmin = 1
    nmax = max(nmin, nonneg_int_floor(nmax_raw))
    n0_vals = np.arange(nmin, nmax + 1, dtype=int)

    for m0 in m0_list:
        p_r, p_rs, p_mc, p_mc_se = [], [], [], []

        for n0 in n0_vals:
            out = pvals_onoff(
                s0,
                tau,
                n0,
                m0,
                sigrel=sigrel,
                min_toys=min_toys,
                max_toys=max_toys,
            )
            p_r.append(out["p_r"])
            p_rs.append(out["p_rstar"])
            p_mc.append(out["p_mc"])
            p_mc_se.append(out["p_mc_se"])

        p_r = np.asarray(p_r, dtype=float)
        p_rs = np.asarray(p_rs, dtype=float)
        p_mc = np.asarray(p_mc, dtype=float)
        p_err = np.asarray(p_mc_se, dtype=float)

        denom = np.clip(p_mc, 1e-16, None)
        rel_r = np.abs(p_r - p_mc) / denom
        rel_rs = np.abs(p_rs - p_mc) / denom

        eps = 1e-16
        p_r = np.maximum(p_r, eps)
        p_rs = np.maximum(p_rs, eps)
        p_mc = np.maximum(p_mc, eps)
        rel_r = np.maximum(rel_r, eps)
        rel_rs = np.maximum(rel_rs, eps)

        fig, (ax_top, ax_bot) = plt.subplots(
            2, 1, figsize=(12, 9), sharex=True,
            gridspec_kw={'height_ratios': [3.5, 1.2]}
        )

        ax_top.semilogy(n0_vals, p_r, marker="o", linestyle="None", ms=5, label="1 − Φ(r)", color="tab:blue")
        ax_top.semilogy(n0_vals, p_rs, marker="^", linestyle="None", ms=5, label="1 − Φ(r*)", color="tab:orange")
        ax_top.errorbar(
            n0_vals, p_mc, yerr=p_err, fmt="x", ms=4, lw=1, capsize=2, label="MC", color="tab:green"
        )
        ax_top.set_ylabel("p-value (upper tail)")
        ax_top.set_xlim(nmin - 0.5, nmax + 0.5)
        ax_top.set_title(fr"$m_0={m0}$,  $s_0={s0}$,  $b={b}$,  $	au={tau}$,  $\sigma_\mathrm{{rel}}={sigrel}$")
        ax_top.grid(True, which="both", alpha=0.25)
        ax_top.legend()

        ax_bot.semilogy(n0_vals, rel_r, marker="o", linestyle="None", ms=4, label=r"|r − MC| / MC", color="tab:blue")
        ax_bot.semilogy(n0_vals, rel_rs, marker="^", linestyle="None", ms=4, label=r"|r* − MC| / MC", color="tab:orange")
        ax_bot.set_xlabel(r"$n_0$ (observed ON counts)")
        ax_bot.set_ylabel("rel. abs. diff")
        ax_bot.grid(True, which="both", alpha=0.25)
        ax_bot.legend()

        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)


def run_pval_pdf():
    out_dir.mkdir(parents=True, exist_ok=True)
    pdf_path = out_dir / pdf_name
    with PdfPages(pdf_path) as pdf:
        for s0, b, tau in product(s_vec, b_vec, tau_vec):
            run_case(float(s0), float(b), float(tau), float(sigrel), pdf)
    print(f"Saved all plots to: {pdf_path.resolve()}")

run_pval_pdf()


In [ ]:

# ---------- On/off expected significance scan (serial analogue of the Condor sweep) ----------
# Outer loop: repeat pseudo-experiments to build the distribution of Z (like many Condor jobs)
# Inner work: for each pseudo-dataset (n_obs, m_obs), compute a p-value via toys and convert to Z

# Scan settings (mirrors parallelization/config.yaml)
s_vec     = np.array([1.0, 2.0, 5.0], dtype=float)
tauVec    = np.array([0.5, 1.0, 2.0], dtype=float)
relSigVec = np.array([0.2, 0.5, 1.0], dtype=float)

# b-range for fixed-tau scans
b_min_tau, b_max_tau, n_bpts_tau = 0.1, 100.0, 30

# b-range for fixed-sigma_rel scans
b_min_sig, b_max_sig, n_bpts_sig = 0.1, 100.0, 30

# Toy controls (match the farm defaults)
sigrel_Z = 0.01
min_toys = 1_000
max_toys = 1_000_000

# Number of pseudo-experiments used to estimate the median significance per point
n_outer_experiments = 150  # set to 1000 to match the Condor run exactly

# Seeds
outer_seed = 12345
inner_seed = 67890

outdir = Path("plots")
outdir.mkdir(exist_ok=True)
summary_pdf_path = outdir / "onoff_medsig.pdf"

b_values_tau = np.logspace(np.log10(b_min_tau), np.log10(b_max_tau), n_bpts_tau)
b_values_sig = np.logspace(np.log10(b_min_sig), np.log10(b_max_sig), n_bpts_sig)


def _fmt(x):
    """Filename-friendly float formatter: 0.1 -> '0p1'."""
    return f"{x:g}".replace(".", "p")


def compute_single_Z(s_true, b, tau, rng_outer, rng_inner):
    """One pseudo-experiment: draw (n_obs, m_obs), compute p_mc, convert to Z."""
    n_obs = int(rng_outer.poisson(lam=s_true + b))
    m_obs = int(rng_outer.poisson(lam=tau * b))
    inner_seed_local = int(rng_inner.integers(0, 2**31 - 1))

    out = pvals_onoff(
        s=0.0,
        tau=tau,
        n=n_obs,
        m=m_obs,
        sigrel=sigrel_Z,
        min_toys=min_toys,
        max_toys=max_toys,
        seed=inner_seed_local,
    )
    p = float(out["p_mc"])
    return norm.isf(p)


def run_outer_experiments(n_outer):
    """Accumulate Z values for each grid point across many pseudo-experiments."""
    Z_groups = defaultdict(list)

    for outer_idx in range(n_outer):
        rng_outer = np.random.default_rng(outer_seed + outer_idx)
        rng_inner = np.random.default_rng(inner_seed + outer_idx)

        for s_idx, s_true in enumerate(s_vec):
            # Fixed-tau scan
            for tau_idx, tau in enumerate(tauVec):
                for b_idx, b in enumerate(b_values_tau):
                    Z_single = compute_single_Z(s_true, b, tau, rng_outer, rng_inner)
                    Z_groups[("tau", s_idx, tau_idx, b_idx)].append(Z_single)

            # Fixed-sigma_rel scan (tau depends on b)
            for sig_idx, sigma_rel in enumerate(relSigVec):
                for b_idx, b in enumerate(b_values_sig):
                    tau_b = 1.0 / (sigma_rel**2 * b)
                    Z_single = compute_single_Z(s_true, b, tau_b, rng_outer, rng_inner)
                    Z_groups[("sig", s_idx, sig_idx, b_idx)].append(Z_single)

    return Z_groups


def median_grids(Z_groups):
    """Return arrays of median Z for each grid point (tau-scan and sigma_rel-scan)."""
    Z_med_tau = np.full((len(s_vec), len(tauVec), len(b_values_tau)), np.nan)
    Z_med_sig = np.full((len(s_vec), len(relSigVec), len(b_values_sig)), np.nan)

    for (mode, s_idx, param_idx, b_idx), values in Z_groups.items():
        if len(values) == 0:
            continue
        med = float(np.median(np.asarray(values, dtype=float)))
        if mode == "tau":
            Z_med_tau[s_idx, param_idx, b_idx] = med
        else:
            Z_med_sig[s_idx, param_idx, b_idx] = med

    return Z_med_tau, Z_med_sig


Z_groups = run_outer_experiments(n_outer_experiments)
Z_med_tau, Z_med_sig = median_grids(Z_groups)

# Asimov curves for each grid point
Z_A_r_tau     = np.zeros((len(s_vec), len(tauVec), len(b_values_tau)), dtype=float)
Z_A_rstar_tau = np.zeros_like(Z_A_r_tau)
Z_A_r_sig     = np.zeros((len(s_vec), len(relSigVec), len(b_values_sig)), dtype=float)
Z_A_rstar_sig = np.zeros_like(Z_A_r_sig)

for s_idx, s_true in enumerate(s_vec):
    for tau_idx, tau in enumerate(tauVec):
        for b_idx, b in enumerate(b_values_tau):
            asim = asimov_Zs_onoff(s_true, b, tau)
            Z_A_r_tau[s_idx, tau_idx, b_idx]     = asim["Z_A_r"]
            Z_A_rstar_tau[s_idx, tau_idx, b_idx] = asim["Z_A_rstar"]

    for sig_idx, sigma_rel in enumerate(relSigVec):
        for b_idx, b in enumerate(b_values_sig):
            tau_b = 1.0 / (sigma_rel**2 * b)
            asim = asimov_Zs_onoff(s_true, b, tau_b)
            Z_A_r_sig[s_idx, sig_idx, b_idx]     = asim["Z_A_r"]
            Z_A_rstar_sig[s_idx, sig_idx, b_idx] = asim["Z_A_rstar"]

with PdfPages(summary_pdf_path) as pdf:
    for s_idx, s_true in enumerate(s_vec):
        # ----- Fixed tau -----
        for tau_idx, tau in enumerate(tauVec):
            fig, ax = plt.subplots(figsize=(8, 5), dpi=150)

            ax.plot(b_values_tau, Z_A_r_tau[s_idx, tau_idx],     label=r"Asimov $r$ (on/off)")
            ax.plot(b_values_tau, Z_A_rstar_tau[s_idx, tau_idx], "--", label=r"Asimov $r^st$ (on/off)")
            ax.plot(b_values_tau, Z_med_tau[s_idx, tau_idx],     linestyle="None", marker="x",
                    label=r"MC median $Z$")

            ax.set_xscale("log")
            ax.set_xlabel(r"$b$")
            ax.set_ylabel(r"$r,\, r^st,\, Z$")
            ax.set_ylim(bottom=-1, top=6)
            ax.grid(True, which="both", ls="--", alpha=0.35)
            ax.set_title(rf"$s_\mathrm{{true}} = {s_true}$, fixed $	au = {tau}$")
            ax.legend(frameon=False, loc="upper right")

            plt.tight_layout()
            fname = outdir / f"onoff_bscan_s{_fmt(s_true)}_tau{_fmt(tau)}.pdf"
            fig.savefig(fname)
            pdf.savefig(fig)
            plt.close(fig)

        # ----- Fixed sigma_rel -----
        for sig_idx, sigma_rel in enumerate(relSigVec):
            fig, ax = plt.subplots(figsize=(8, 5), dpi=150)

            ax.plot(b_values_sig, Z_A_r_sig[s_idx, sig_idx],     label=r"Asimov $r$ (on/off)")
            ax.plot(b_values_sig, Z_A_rstar_sig[s_idx, sig_idx], "--", label=r"Asimov $r^st$ (on/off)")
            ax.plot(b_values_sig, Z_med_sig[s_idx, sig_idx],     linestyle="None", marker="x",
                    label=r"MC median $Z$")

            ax.set_xscale("log")
            ax.set_xlabel(r"$b$")
            ax.set_ylabel(r"$r,\, r^st,\, Z$")
            ax.set_ylim(bottom=-1, top=8)
            ax.grid(True, which="both", ls="--", alpha=0.35)
            ax.set_title(
                rf"$s_\mathrm{{true}} = {s_true}$, "
                rf"fixed $\sigma_b/b = {sigma_rel}$ "
                r"(i.e. $	au(b)=1/(\sigma_\mathrm{rel}^2\,b)$)"
            )
            ax.legend(frameon=False, loc="upper right")

            plt.tight_layout()
            fname = outdir / f"onoff_bscan_s{_fmt(s_true)}_sigrel{_fmt(sigma_rel)}.pdf"
            fig.savefig(fname)
            pdf.savefig(fig)
            plt.close(fig)

print(f"Saved individual plots and combined PDF to {summary_pdf_path.resolve()}")
